# Phase-1: Data Exploration - NASA B0005 Battery

**Goal:** Load, inspect, and understand the B0005 battery dataset.

**Steps:**
1. Load battery data from .mat file
2. Inspect cycle structure
3. Visualize capacity degradation
4. Examine voltage, current, temperature profiles
5. Apply cycle filtering
6. Analyze data quality

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils import load_config, plot_style_setup
from data_loader import load_battery_data
from cycle_segmentation import segment_and_filter_cycles

# Setup plotting style
plot_style_setup()

# Load configuration
config = load_config('config.yaml')

print("✓ Imports complete")

## 1. Load Battery Data

In [ ]:
# Load B0005 battery data
loader = load_battery_data(config)

## 2. Extract Discharge Cycles

In [ ]:
# Get discharge cycles (used for SOH)
discharge_cycles = loader.get_discharge_cycles()

print(f"\nTotal discharge cycles: {len(discharge_cycles)}")

## 3. Visualize Capacity Degradation

In [ ]:
# Extract capacities
capacities = [c.capacity for c in discharge_cycles if c.capacity is not None]
cycle_numbers = list(range(len(capacities)))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cycle_numbers, capacities, 'o-', linewidth=2, markersize=4)
ax.axhline(y=2.0, color='g', linestyle='--', label='Rated Capacity (2.0 Ah)')
ax.axhline(y=1.4, color='r', linestyle='--', label='EOL (1.4 Ah, 30% fade)')
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('Discharge Capacity (Ah)', fontsize=12)
ax.set_title('Battery B0005: Capacity Degradation Over Cycles', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial capacity: {capacities[0]:.4f} Ah")
print(f"Final capacity: {capacities[-1]:.4f} Ah")
print(f"Capacity fade: {(capacities[0] - capacities[-1]):.4f} Ah ({(capacities[0] - capacities[-1])/capacities[0]*100:.2f}%)")

## 4. Examine Sample Discharge Profiles

In [ ]:
# Select cycles to visualize: early, middle, late
n_cycles = len(discharge_cycles)
sample_indices = [0, n_cycles//2, n_cycles-1]
sample_labels = ['Early (Cycle 0)', f'Middle (Cycle {n_cycles//2})', f'Late (Cycle {n_cycles-1})']
colors = ['green', 'orange', 'red']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for idx, label, color in zip(sample_indices, sample_labels, colors):
    cycle = discharge_cycles[idx]
    
    # Voltage vs Time
    axes[0, 0].plot(cycle.time, cycle.voltage, label=label, color=color, linewidth=2)
    
    # Current vs Time
    axes[0, 1].plot(cycle.time, cycle.current, label=label, color=color, linewidth=2)
    
    # Temperature vs Time
    axes[1, 0].plot(cycle.time, cycle.temperature, label=label, color=color, linewidth=2)
    
    # Voltage vs Capacity (discharge curve)
    # Compute cumulative capacity
    dt = np.diff(cycle.time, prepend=cycle.time[0])
    capacity_curve = np.cumsum(np.abs(cycle.current) * dt) / 3600.0
    axes[1, 1].plot(capacity_curve, cycle.voltage, label=label, color=color, linewidth=2)

# Configure subplots
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Voltage (V)')
axes[0, 0].set_title('Voltage Profile')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Current (A)')
axes[0, 1].set_title('Current Profile')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Temperature (°C)')
axes[1, 0].set_title('Temperature Profile')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].set_xlabel('Capacity (Ah)')
axes[1, 1].set_ylabel('Voltage (V)')
axes[1, 1].set_title('Discharge Curve (V vs Q)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Battery B0005: Discharge Profile Comparison', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 5. Apply Cycle Filtering

In [ ]:
# Apply filtering criteria from Phase-1 decisions
filtered_cycles = segment_and_filter_cycles(loader, config)

print(f"\n✓ Filtered cycles: {len(filtered_cycles)} / {len(discharge_cycles)}")

## 6. Data Quality Analysis

In [ ]:
# Analyze data point distribution
data_points = [c.data_points for c in filtered_cycles]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of data points
axes[0].hist(data_points, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(x=np.mean(data_points), color='r', linestyle='--', label=f'Mean: {np.mean(data_points):.0f}')
axes[0].set_xlabel('Data Points per Cycle')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Data Points')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Data points over cycles
axes[1].plot(data_points, 'o-', markersize=3)
axes[1].set_xlabel('Cycle Number')
axes[1].set_ylabel('Data Points')
axes[1].set_title('Data Points per Cycle Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Temperature Analysis

In [ ]:
# Analyze temperature variation (ΔT) per cycle
temp_deltas = [c.temperature.max() - c.temperature.min() for c in filtered_cycles]
temp_means = [c.temperature.mean() for c in filtered_cycles]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Temperature delta over cycles
axes[0].plot(temp_deltas, 'o-', markersize=3, color='orange')
axes[0].axhline(y=np.mean(temp_deltas), color='r', linestyle='--', label=f'Mean ΔT: {np.mean(temp_deltas):.2f}°C')
axes[0].set_xlabel('Cycle Number')
axes[0].set_ylabel('ΔT (°C)')
axes[0].set_title('Temperature Variation per Cycle')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean temperature over cycles
axes[1].plot(temp_means, 'o-', markersize=3, color='green')
axes[1].axhline(y=np.mean(temp_means), color='r', linestyle='--', label=f'Mean: {np.mean(temp_means):.2f}°C')
axes[1].set_xlabel('Cycle Number')
axes[1].set_ylabel('Mean Temperature (°C)')
axes[1].set_title('Mean Temperature per Cycle')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTemperature Statistics:")
print(f"  Mean ΔT: {np.mean(temp_deltas):.2f} ± {np.std(temp_deltas):.2f} °C")
print(f"  Mean temperature: {np.mean(temp_means):.2f} ± {np.std(temp_means):.2f} °C")
print(f"\n  Note: Low variance expected for room temperature testing")

## Summary

**Key Findings:**
1. B0005 has clean, complete discharge cycles
2. Clear capacity degradation trend from ~2.0 Ah to ~1.4 Ah
3. Voltage profiles show consistent discharge behavior
4. Temperature is stable (room temperature testing)
5. Data quality is high after filtering

**Next Steps:**
- Extract ICA features (see notebook 02)
- Extract all physics features
- Train baseline model (see notebook 03)